# Reasoning Segmentation - Final

**팀명**: Team 7   
**Seed**: 428 (frozen)

대회 제출 전용. 전체 test set → `outputs/submission.csv`

# 1. 환경 설정

In [ ]:
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from src.config import FINAL_CONFIG, TEST_CSV, DATASET_DIR
from src.utils import set_seed, rle_encode, load_image
from src.models import ModelManager, Qwen3Model
from src.multi import is_multi_object_query
from pipeline.v4_final import run_final

OUTPUT_DIR = FINAL_CONFIG['output_dir']
SUBMISSION_FILE = FINAL_CONFIG['submission_file']
SEED = FINAL_CONFIG['seed']

print(f"PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"Output: {OUTPUT_DIR}/{SUBMISSION_FILE}")

In [ ]:
set_seed(SEED)
Path(OUTPUT_DIR).mkdir(exist_ok=True)
print(f"Seed: {SEED}")

In [ ]:
test_df = pd.read_csv(TEST_CSV)
test_df['is_multi'] = test_df['query'].apply(is_multi_object_query)

print(f"Total samples: {len(test_df)}")
print(f"Single: {(~test_df['is_multi']).sum()}, Multi: {test_df['is_multi'].sum()}")

# 2. 실행

In [ ]:
model_manager = ModelManager()
model_manager.load_sam()

qwen3 = Qwen3Model()
qwen3.load()
print("Models loaded.")

In [ ]:
all_data = []

for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Loading images"):
    image_path = Path(DATASET_DIR) / "test" / f"{row['ID']}.jpg"
    all_data.append({
        'id': row['ID'],
        'image': load_image(str(image_path)),
        'query': row['query'],
        'mllm_pred': None,
        'sam_mask': None,
    })

print(f"Loaded {len(all_data)} images")

In [ ]:
run_final(
    mllm_model=qwen3,
    sam_model=model_manager.sam,
    all_data=all_data,
)

In [ ]:
submission = [
    {'ID': item['id'], 'Label': rle_encode(item['sam_mask'])}
    for item in all_data
]

submission_df = pd.DataFrame(submission)
assert submission_df['ID'].tolist() == test_df['ID'].tolist(), "Order mismatch!"

submission_path = Path(OUTPUT_DIR) / SUBMISSION_FILE
submission_df.to_csv(submission_path, index=False)

print(f"Submission saved: {submission_path}")
print(f"Total samples: {len(submission_df)}")